In [2]:
pip install imageio-ffmpeg librosa soundfile numpy pandas

Note: you may need to restart the kernel to use updated packages.


In [ ]:
# ===== Step 0: imports =====
import io
import os
import subprocess

import imageio_ffmpeg
import librosa
import numpy as np
import pandas as pd
import soundfile as sf

# ===== Step 1: paths =====
audio_folder = r"D:\CHIRP\data\noise_m4a"
output_csv = r"D:\CHIRP\analysis\outputs\noise_features.csv"

# ===== Step 2: load m4a (librosa needs ffmpeg for m4a on Windows) =====
def load_m4a(file_path, sr=22050):
    ffmpeg = imageio_ffmpeg.get_ffmpeg_exe()
    cmd = [
        ffmpeg,
        "-nostdin",
        "-i",
        file_path,
        "-f",
        "wav",
        "-acodec",
        "pcm_s16le",
        "-ac",
        "1",
        "-ar",
        str(sr),
        "pipe:1",
    ]
    result = subprocess.run(cmd, capture_output=True, check=True)
    y, sr_out = sf.read(io.BytesIO(result.stdout))
    y = np.asarray(y, dtype=np.float32)
    if y.ndim > 1:
        y = y.mean(axis=1)
    return y, sr_out

# ===== Step 3: split into 1.5 s clips (non-overlapping) =====
CLIP_DURATION = 1.5

def iter_segments(y, sr, clip_duration=CLIP_DURATION):
    """Yield non-overlapping clip_duration-second chunks (full clips only)."""
    n_samples = int(clip_duration * sr)
    if n_samples <= 0 or len(y) < n_samples:
        return
    for start in range(0, len(y) - n_samples + 1, n_samples):
        yield y[start : start + n_samples]

# ===== Step 4: feature extraction (same as 01_audio_feature_extraction) =====
def extract_features_from_audio(y, sr=22050, n_mfcc=13):
    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
    mfcc_mean = mfccs.mean(axis=1)
    mfcc_std = mfccs.std(axis=1)

    chroma = librosa.feature.chroma_stft(y=y, sr=sr)
    chroma_mean = chroma.mean(axis=1)

    zcr = librosa.feature.zero_crossing_rate(y)
    zcr_mean = zcr.mean()

    rms = librosa.feature.rms(y=y)
    rms_mean = rms.mean()

    spec_cent = librosa.feature.spectral_centroid(y=y, sr=sr)
    spec_cent_mean = spec_cent.mean()

    return np.concatenate([
        mfcc_mean, mfcc_std, chroma_mean,
        [zcr_mean, rms_mean, spec_cent_mean],
    ])

# ===== Step 5: load each m4a, clip, extract features =====
m4a_files = sorted(f for f in os.listdir(audio_folder) if f.lower().endswith(".m4a"))
print(f"Found {len(m4a_files)} m4a files in {audio_folder}")

feature_list = []
file_names = []
segment_indices = []

for fname in m4a_files:
    path = os.path.join(audio_folder, fname)
    base = os.path.splitext(fname)[0]
    y, sr = load_m4a(path, sr=22050)
    n_clips = 0
    for seg_idx, segment in enumerate(iter_segments(y, sr)):
        feature_list.append(extract_features_from_audio(segment, sr=sr))
        file_names.append(base)
        segment_indices.append(seg_idx)
        n_clips += 1
    print(f"  {fname}: {len(y) / sr:.2f}s -> {n_clips} clip(s)")

print(f"Total clips: {len(feature_list)}")

# ===== Step 6: build DataFrame =====
mfcc_cols = [f"mfcc{i+1}_mean" for i in range(13)] + [f"mfcc{i+1}_std" for i in range(13)]
chroma_cols = [f"chroma{i+1}" for i in range(12)]
other_cols = ["zcr", "rms", "spectral_centroid"]
all_feature_cols = mfcc_cols + chroma_cols + other_cols

features_df = pd.DataFrame(np.array(feature_list), columns=all_feature_cols)
metadata_df = pd.DataFrame({
    "Number": range(1, len(file_names) + 1),
    "FileName": file_names,
    "Segment": segment_indices,
})
full_df = pd.concat([metadata_df, features_df], axis=1)

# ===== Step 7: save =====
full_df.to_csv(output_csv, index=False, encoding="utf-8")
print(f"Feature extraction completed! Saved to {output_csv}")
print(full_df.head())

NameError: name 'ffmpeg' is not defined